In [1]:
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.4 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [9]:
fr_path = '/content/drive/MyDrive/cs505_data/europarl-v7.fr-en.fr'
en_path = '/content/drive/MyDrive/cs505_data/europarl-v7.fr-en.en'

with open(fr_path, encoding='utf-8') as f:
  fr_lines = f.readlines()
with open(en_path, encoding='utf-8') as f:
  en_lines = f.readlines()

print(f"Total sentence pairs: {len(fr_lines):,}")
print(f"FR: {fr_lines[0].strip()}")
print(f"EN: {en_lines[0].strip()}")

Total sentence pairs: 2,007,723
FR: Reprise de la session
EN: Resumption of the session


In [10]:
from collections import Counter
import math
import re
import sacrebleu

def tokenize(sentence):
  return sentence.lower().strip().split()

# Keep this aligned with your other notebooks for fair comparisons.
TRAIN_SIZE = 50000
train_fr = [tokenize(s) for s in fr_lines[:TRAIN_SIZE]]
train_en = [tokenize(s) for s in en_lines[:TRAIN_SIZE]]

print(f"Train pairs used: {len(train_fr):,}")
print(f"Sample FR tokens: {train_fr[0]}")
print(f"Sample EN tokens: {train_en[0]}")

Train pairs used: 50,000
Sample FR tokens: ['reprise', 'de', 'la', 'session']
Sample EN tokens: ['resumption', 'of', 'the', 'session']


In [15]:
PHRASE_MAX_LEN = 4
MIN_PHRASE_COUNT = 2
TOP_K_TRANSLATIONS = 5

def build_fallback_lexicon(fr_corpus, en_corpus):
  counts = {}
  for fr_sent, en_sent in zip(fr_corpus, en_corpus):
    lf, le = len(fr_sent), len(en_sent)
    if lf == 0 or le == 0:
      continue
    ratio = le / lf
    for i, fr_word in enumerate(fr_sent):
      j = min(le - 1, max(0, int(round(i * ratio))))
      en_word = en_sent[j]
      if fr_word not in counts:
        counts[fr_word] = Counter()
      counts[fr_word][en_word] += 1

  fallback = {}
  for fr_word, counter in counts.items():
    fallback[fr_word] = counter.most_common(1)[0][0]
  return fallback

def build_phrase_table(fr_corpus, en_corpus, max_phrase_len=3, min_count=2, top_k=4):
  phrase_counts = {}

  for fr_sent, en_sent in zip(fr_corpus, en_corpus):
    lf, le = len(fr_sent), len(en_sent)
    if lf == 0 or le == 0:
      continue

    ratio = le / lf
    for i in range(lf):
      for span in range(1, max_phrase_len + 1):
        if i + span > lf:
          break

        j0 = int(round(i * ratio))
        j1 = int(round((i + span) * ratio))
        if j1 <= j0:
          j1 = j0 + 1
        if j1 > le:
          continue
        if (j1 - j0) > max_phrase_len:
          continue

        fr_phrase = tuple(fr_sent[i:i + span])
        en_phrase = tuple(en_sent[j0:j1])

        if fr_phrase not in phrase_counts:
          phrase_counts[fr_phrase] = Counter()
        phrase_counts[fr_phrase][en_phrase] += 1

  phrase_table = {}
  for fr_phrase, en_counter in phrase_counts.items():
    filtered = [(en_phrase, c) for en_phrase, c in en_counter.items() if c >= min_count]
    if not filtered:
      continue

    total = sum(c for _, c in filtered)
    ranked = sorted(
        ((en_phrase, c / total) for en_phrase, c in filtered),
        key=lambda x: x[1],
        reverse=True,
    )[:top_k]
    phrase_table[fr_phrase] = ranked

  return phrase_table

def build_bigram_lm(en_corpus):
  unigram = Counter()
  bigram = Counter()

  for sent in en_corpus:
    seq = ['<s>'] + sent + ['</s>']
    for token in seq:
      unigram[token] += 1
    for i in range(1, len(seq)):
      bigram[(seq[i - 1], seq[i])] += 1

  vocab_size = len(unigram)

  def sentence_logprob(tokens):
    seq = ['<s>'] + list(tokens) + ['</s>']
    lp = 0.0
    for i in range(1, len(seq)):
      prev_t = seq[i - 1]
      curr_t = seq[i]
      lp += math.log((bigram[(prev_t, curr_t)] + 1.0) / (unigram[prev_t] + vocab_size))
    return lp

  return sentence_logprob

fallback_lexicon = build_fallback_lexicon(train_fr, train_en)
phrase_table = build_phrase_table(
    train_fr,
    train_en,
    max_phrase_len=PHRASE_MAX_LEN,
    min_count=MIN_PHRASE_COUNT,
    top_k=TOP_K_TRANSLATIONS,
)
lm_score_fn = build_bigram_lm(train_en)

print(f"Fallback lexicon entries: {len(fallback_lexicon):,}")
print(f"Phrase table entries: {len(phrase_table):,}")

Fallback lexicon entries: 54,838
Phrase table entries: 43,097


In [16]:
def phrase_decode(fr_sentence, phrase_table, lm_score_fn, fallback_lexicon, max_phrase_len=3, beam_size=12, lm_weight=0.35):
  src_len = len(fr_sentence)
  if src_len == 0:
    return []

  hyps = [(0, 0.0, [])]
  while True:
    active = [h for h in hyps if h[0] < src_len]
    if not active:
      break

    expanded = []
    for pos, score, out_tokens in hyps:
      if pos >= src_len:
        expanded.append((pos, score, out_tokens))
        continue

      for span in range(1, max_phrase_len + 1):
        if pos + span > src_len:
          break

        fr_phrase = tuple(fr_sentence[pos:pos + span])
        candidates = phrase_table.get(fr_phrase)
        if not candidates and span == 1:
          fr_word = fr_phrase[0]
          if fr_word in fallback_lexicon:
            candidates = [((fallback_lexicon[fr_word],), 1e-6)]
          else:
            # Avoid <UNK> to preserve named entities/acronyms.
            candidates = [((fr_word,), 1e-7)]

        if not candidates:
          continue

        for en_phrase, p in candidates:
          new_score = score + math.log(max(p, 1e-12))
          new_out = out_tokens + list(en_phrase)
          expanded.append((pos + span, new_score, new_out))

    hyps = sorted(expanded, key=lambda x: (x[0], x[1]), reverse=True)[:beam_size]

  complete = [h for h in hyps if h[0] == src_len]
  if not complete:
    return max(hyps, key=lambda x: (x[0], x[1]))[2]
  return max(complete, key=lambda h: h[1] + lm_weight * lm_score_fn(h[2]))[2]

In [17]:
#load and parse WMT14 test data
def parse_sgm(filepath):
  sentences = []
  with open(filepath, encoding='utf-8') as f:
    for line in f:
      line = line.strip()
      if line.startswith('<seg'):
        match = re.search(r'<seg[^>]*>(.*?)</seg>', line)
        if match:
          sentences.append(match.group(1).strip())
  return sentences

fr_test_path = '/content/drive/MyDrive/cs505_data/newstest2014-fren-src.fr.sgm'
en_test_path = '/content/drive/MyDrive/cs505_data/newstest2014-fren-ref.en.sgm'

test_fr = parse_sgm(fr_test_path)
test_en = parse_sgm(en_test_path)
test_fr_tok = [tokenize(s) for s in test_fr]
test_en_tok = [tokenize(s) for s in test_en]

print(f"Test sentences: {len(test_fr)}")
print(f"Sample FR tokens: {test_fr_tok[0]}")
print(f"Sample EN tokens: {test_en_tok[0]}")

Test sentences: 3003
Sample FR tokens: ["l'affaire", 'nsa', 'souligne', "l'absence", 'totale', 'de', 'débat', 'sur', 'le', 'renseignement']
Sample EN tokens: ['nsa', 'affair', 'emphasizes', 'complete', 'lack', 'of', 'debate', 'on', 'intelligence']


In [19]:
# Evaluate phrase-based system with SacreBLEU
phrase_translations = []
for idx, fr_tokens in enumerate(test_fr_tok):
    pred = phrase_decode(
        fr_tokens,
        phrase_table,
        lm_score_fn,
        fallback_lexicon,
        max_phrase_len=PHRASE_MAX_LEN,
        beam_size=12,
        lm_weight=0.35,
    )
    phrase_translations.append(' '.join(pred))
    if (idx + 1) % 500 == 0:
        print(f'Decode progress: {idx + 1}/{len(test_fr_tok)}')

phrase_bleu = sacrebleu.corpus_bleu(phrase_translations, [test_en])
print(f'Phrase-based BLEU score: {phrase_bleu.score:.2f}')

print('\nSamples:')
for i in range(3):
    print(f'FR : {test_fr[i]}')
    print(f'REF: {test_en[i]}')
    print(f'PB : {phrase_translations[i]}')
    print()

Decode progress: 500/3003
Decode progress: 1000/3003
Decode progress: 1500/3003
Decode progress: 2000/3003
Decode progress: 2500/3003
Decode progress: 3000/3003
Phrase-based BLEU score: 5.22

Samples:
FR : L'affaire NSA souligne l'absence totale de débat sur le renseignement
REF: NSA Affair Emphasizes Complete Lack of Debate on Intelligence
PB : the nsa in the of the debate on the intelligence

FR : Comment expliquer l'attitude contradictoire du gouvernement français, qui d'un coté s'offusque en public en convoquant l'ambassadeur des Etats-Unis le 21 octobre, et de l'autre interdit le survol du territoire par l'avion présidentiel bolivien, sur la base de la rumeur de la présence à son bord d'Edward Snowden ?
REF: Why the contradictory attitude of the French government? On the one hand, it publicly takes offence and summons the Ambassador of the United States on October 21 and, on the other, it forbids the Bolivian president's plane to enter its air space on the basis of a rumor that Ed